# ECS271 RAG Project — RunPod

Adapted from Colab. All paths use `/workspace` instead of `/content/drive/MyDrive/ecs271`.
Before starting JupyterLab, set your HF token: `export HF_TOKEN=hf_...`
rclone config must be at `/workspace/rclone.conf` with remote named `gdrive`.

In [ ]:
import os, subprocess

if not os.path.exists('/workspace/ECS271Project/.git'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/lichunho/ECS271Project.git', '/workspace/ECS271Project'],
        check=True
    )
os.chdir('/workspace/ECS271Project')
print('cwd:', os.getcwd())

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -U "numpy<2.1" transformers accelerate sentencepiece python-dotenv openai httpx tqdm huggingface_hub

In [ ]:
!pip uninstall -y vllm
!rm -rf /usr/local/lib/python3.12/dist-packages/vllm*
!rm -rf /usr/local/lib/python3.12/dist-packages/*vllm*
!uv cache clean
!uv pip install --system --pre -U vllm \
  --extra-index-url https://wheels.vllm.ai/nightly/cu129
!pip install -q -U "numpy<2.1" openai httpx tqdm transformers accelerate sentencepiece huggingface_hub

In [ ]:
import torch, sys, vllm

print('python', sys.version)
print('torch', torch.__version__)
print('torch cuda', torch.version.cuda)
print('vllm', vllm.__version__)

## Auth

In [ ]:
from huggingface_hub import login
import os

hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print('logged in via HF_TOKEN')
else:
    login()

## Get Data from Drive

Route predictions are already complete — pull them down before running Phase 2.
Model checkpoint cells are only needed if re-running Phase 1.

In [ ]:
!mkdir -p /workspace/route_predictions
!rclone --config /workspace/rclone.conf copy gdrive:route_predictions /workspace/route_predictions --progress
!ls -lh /workspace/route_predictions/

In [ ]:
# Only needed if re-running predict_routes
!rclone --config /workspace/rclone.conf copy gdrive:deberta-v3-base-binary-silver /workspace/deberta-v3-base-binary-silver --progress
!rclone --config /workspace/rclone.conf copy gdrive:deberta-v3-base-binary-silver-balanced-weighted /workspace/deberta-v3-base-binary-silver-balanced-weighted --progress
!rclone --config /workspace/rclone.conf copy gdrive:t5-large-binary-silver-fp32 /workspace/t5-large-binary-silver-fp32 --progress

## vLLM Server

In [ ]:
import subprocess, time, requests

log_path = '/workspace/vllm.log'
vllm_log = open(log_path, 'w')

vllm_proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'google/gemma-4-26B-A4B',
    '--served-model-name', 'google/gemma-4-26b-a4b',
    '--host', '0.0.0.0',
    '--port', '8000',
    '--dtype', 'bfloat16',
    '--gpu-memory-utilization', '0.88',
    '--max-model-len', '4096',
], stdout=vllm_log, stderr=subprocess.STDOUT)

for i in range(120):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.ok:
            print('vLLM ready:', r.json())
            break
    except Exception:
        pass
    time.sleep(5)
else:
    print('vLLM did not start in time. Check /workspace/vllm.log')
    !tail -120 /workspace/vllm.log

In [ ]:
# logprobs smoke test
import requests, json

payload = {
    'model': 'google/gemma-4-26b-a4b',
    'prompt': 'Q: What is the speed of a swallow?\nA:',
    'max_tokens': 20,
    'temperature': 0,
    'logprobs': 0,
}

r = requests.post('http://localhost:8000/v1/completions', json=payload, timeout=120)
print('status:', r.status_code)
print(r.text[:2000])

if r.ok:
    data = r.json()
    choice = data['choices'][0]
    print('has logprobs:', choice.get('logprobs') is not None)
    print(json.dumps(choice, indent=2)[:3000])

## (Optional) Phase 1: Predict Routes

Already completed in Colab. Only re-run if route prediction files are missing.

In [ ]:
# Skip if already trained
!python -m scripts.train_t5_classifier \
  --train-file data/labeled/classifier_train_binary_silver.jsonl \
  --validation-file data/labeled/classifier_valid_silver.jsonl \
  --model-name t5-large \
  --output-dir /workspace/t5-large-binary-silver-fp32 \
  --epochs 15 \
  --batch-size 8 \
  --eval-batch-size 32 \
  --gradient-accumulation-steps 4 \
  --learning-rate 3e-5 \
  --max-length 384 \
  --selection-metric accuracy \
  --no-epoch-checkpoints

In [ ]:
!rm -rf /workspace/route_predictions
!mkdir -p /workspace/route_predictions

In [ ]:
# classify test set — deberta unbalanced
!python -m scripts.predict_routes \
  --data-dir data/eval_500 \
  --model-kind encoder \
  --model-path /workspace/deberta-v3-base-binary-silver \
  --classifier-name deberta-v3-base-binary-silver \
  --output-dir /workspace/route_predictions \
  --batch-size 32 \
  --max-length 384 \
  --include-answers

In [ ]:
# classify test set — deberta balanced
!python -m scripts.predict_routes \
  --data-dir data/eval_500 \
  --model-kind encoder \
  --model-path /workspace/deberta-v3-base-binary-silver-balanced-weighted \
  --classifier-name deberta-v3-base-binary-silver-balanced-weighted \
  --output-dir /workspace/route_predictions \
  --batch-size 32 \
  --max-length 384 \
  --include-answers

In [ ]:
# classify test set — t5
!python -m scripts.predict_routes \
  --data-dir data/eval_500 \
  --model-kind t5 \
  --model-path /workspace/t5-large-binary-silver-fp32 \
  --classifier-name t5-large-binary-silver-fp32 \
  --output-dir /workspace/route_predictions \
  --batch-size 32 \
  --max-length 384 \
  --include-answers

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

route_dir = Path('/workspace/route_predictions')

files = [
    route_dir / 'deberta-v3-base-binary-silver.routes.jsonl',
    route_dir / 'deberta-v3-base-binary-silver-balanced-weighted.routes.jsonl',
    route_dir / 't5-large-binary-silver-fp32.routes.jsonl',
]

for path in files:
    print('=' * 80)
    print(path.name)
    rows = [json.loads(line) for line in path.open()]
    print('rows:', len(rows))
    route_counts = Counter(row['initial_route'] for row in rows)
    print('route counts:', dict(route_counts))
    by_dataset = defaultdict(Counter)
    latencies = []
    for row in rows:
        by_dataset[row['dataset']][row['initial_route']] += 1
        if 'classifier_inference_ms' in row:
            latencies.append(row['classifier_inference_ms'])
    print('by dataset:')
    for dataset in sorted(by_dataset):
        print(' ', dataset, dict(by_dataset[dataset]))
    if latencies:
        print('mean classifier_inference_ms:', sum(latencies) / len(latencies))
    print('sample row:')
    sample = rows[0]
    print(json.dumps({
        'question_id': sample['question_id'],
        'dataset': sample['dataset'],
        'initial_route': sample['initial_route'],
        'route_probabilities': sample['route_probabilities'],
        'classifier_inference_ms': sample.get('classifier_inference_ms'),
    }, indent=2))

## Phase 2: Probe Multi-Routes

In [ ]:
# debug run — limit 20
!rm -f /workspace/probed_routes/debug.probed.jsonl
!mkdir -p /workspace/probed_routes
!python -m scripts.probe_multi_routes \
  --input-routes /workspace/route_predictions/deberta-v3-base-binary-silver.routes.jsonl \
  --output-file /workspace/probed_routes/debug.probed.jsonl \
  --model google/gemma-4-26b-a4b \
  --base-url http://localhost:8000/v1 \
  --api-key EMPTY \
  --api-mode completions \
  --threshold 0.5 \
  --max-tokens 48 \
  --temperature 0 \
  --limit 20

In [ ]:
!head -n 1 /workspace/probed_routes/debug.probed.jsonl | python -m json.tool | head -120
!cat /workspace/probed_routes/debug.probed.summary.json

In [ ]:
# full run — deberta unbalanced
!python -m scripts.probe_multi_routes \
  --input-routes /workspace/route_predictions/deberta-v3-base-binary-silver.routes.jsonl \
  --output-file /workspace/probed_routes/deberta-v3-base-binary-silver.probed.jsonl \
  --model google/gemma-4-26b-a4b \
  --base-url http://localhost:8000/v1 \
  --api-key EMPTY \
  --api-mode completions \
  --threshold 0.5 \
  --max-tokens 48 \
  --temperature 0 \
  --resume

In [ ]:
# full run — deberta balanced
!python -m scripts.probe_multi_routes \
  --input-routes /workspace/route_predictions/deberta-v3-base-binary-silver-balanced-weighted.routes.jsonl \
  --output-file /workspace/probed_routes/deberta-v3-base-binary-silver-balanced-weighted.probed.jsonl \
  --model google/gemma-4-26b-a4b \
  --base-url http://localhost:8000/v1 \
  --api-key EMPTY \
  --api-mode completions \
  --threshold 0.5 \
  --max-tokens 48 \
  --temperature 0 \
  --resume

In [ ]:
# full run — t5
!python -m scripts.probe_multi_routes \
  --input-routes /workspace/route_predictions/t5-large-binary-silver-fp32.routes.jsonl \
  --output-file /workspace/probed_routes/t5-large-binary-silver-fp32.probed.jsonl \
  --model google/gemma-4-26b-a4b \
  --base-url http://localhost:8000/v1 \
  --api-key EMPTY \
  --api-mode completions \
  --threshold 0.5 \
  --max-tokens 48 \
  --temperature 0 \
  --resume

In [ ]:
!wc -l /workspace/probed_routes/*.probed.jsonl
!cat /workspace/probed_routes/deberta-v3-base-binary-silver.probed.summary.json
!cat /workspace/probed_routes/deberta-v3-base-binary-silver-balanced-weighted.probed.summary.json
!cat /workspace/probed_routes/t5-large-binary-silver-fp32.probed.summary.json

## Upload Results to Drive

In [ ]:
!rclone --config /workspace/rclone.conf copy /workspace/probed_routes gdrive:probed_routes --progress